# Phase 5 - Research Rigor Suite

Walks through every Phase-5 module against the shipped artefacts. Runs in ~1 minute on CPU.

The headline claims this notebook substantiates:

1. Cross-dataset leakage probe (NASA vs Mendeley, ROC-AUC on raw features).
2. Calibration / reliability diagram on the bootstrap ensemble.
3. Weighted conformal under NASA -> Mendeley covariate shift.
4. Synthetic right-censored Weibull evaluation.
5. Observational vs causal (interventional) Shapley.
6. DiCE-style counterfactuals for low-SOH cycles.
7. Failure-mode gallery with SHAP attribution.
8. codecarbon wrap-up (total CO2 for the whole suite).

All numbers land under `results/phase5/` as JSON sidecars; we also write a manifest.json summarising everything.

In [ ]:
import json, os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'src' / 'phase5').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Could not locate project root from cwd')
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)

## Option A - run the whole suite at once

Equivalent to `make rigor`. Skip this cell if you want to step through modules one-by-one below.

In [ ]:
from src.phase5 import run_all
manifest = run_all.run_all()
print('Total runtime  :', manifest['total_duration_sec'], 's')
print('Total CO2 (kg) :', manifest['compute']['total_emissions_kg_co2'])
for k, meta in manifest['steps'].items():
    print(f'  {k:<22} {meta["status"]:<10} ({meta["duration_sec"]}s)')

## 1. Leakage probe

In [ ]:
from src.phase5 import leakage
results = leakage.run_project_probe()
print(leakage._format_report(results))

## 2. Calibration + reliability diagram

Tests the bootstrap ensemble against the held-out test split. Generates a reliability diagram PNG under `results/phase5/`.

In [ ]:
from src.phase5 import calibration
calib = calibration.run_project_calibration()
print(calibration._format(calib))

In [ ]:
# Render the reliability diagram inline
from IPython.display import Image
img_path = PROJECT_ROOT / 'results' / 'phase5' / 'reliability_diagram.png'
if img_path.exists():
    display(Image(filename=str(img_path)))

## 3. Weighted conformal under shift

In [ ]:
from src.phase5 import weighted_conformal
wc = weighted_conformal.demo_nasa_to_mendeley()
print(weighted_conformal._format(wc))

## 4. Synthetic-censored Weibull

In [ ]:
from src.phase5 import survival_synth
surv = survival_synth.run_project_survival_synth()
print(survival_synth._format(surv))

## 5. Observational vs causal Shapley

In [ ]:
from src.phase5 import causal_shap
cs = causal_shap.run_project_causal_shap()
print(causal_shap._format(cs))

## 6. Counterfactual explanations

In [ ]:
from src.phase5 import counterfactuals
cf = counterfactuals.run_project_counterfactuals()
print(counterfactuals._format(cf))

## 7. Failure-mode gallery

In [ ]:
from src.phase5 import failure_gallery
fg = failure_gallery.build_failure_gallery()
print(failure_gallery._format(fg))

In [ ]:
# Render the top-3 worst-residual panel
img_path = PROJECT_ROOT / 'results' / 'phase5' / 'failure_gallery.png'
if img_path.exists():
    display(Image(filename=str(img_path)))

## 8. Compute + carbon summary

In [ ]:
from src.phase5 import carbon
agg = carbon.aggregate_reports(PROJECT_ROOT / 'results' / 'phase5' / 'compute')
print(f"Runs tracked    : {agg['n_runs']}")
print(f"Total duration  : {agg.get('total_duration_sec')} s")
print(f"Total energy    : {agg.get('total_energy_kwh')} kWh")
print(f"Total emissions : {agg.get('total_emissions_kg_co2')} kg CO2")

Raw results sit under `results/phase5/*.json`; the human-readable summary is in `PHASE5_RIGOR_REPORT.md`.